# Notebook 09: Parametrización y Reutilización

**Duración**: 40 minutos | **Nivel**: Intermedio

## Introducción

Aprende a crear expectativas reutilizables y parametrizadas para múltiples datasets.

### Objetivos:
1. Parametrizar expectativas
2. Crear suites reutilizables
3. Aplicar misma suite a múltiples datasets
4. Usar variables y configuraciones

In [ ]:
import great_expectations as gx
import pandas as pd

# Cargar múltiples datasets
df_ventas = pd.read_csv("../data/ventas_sucias.csv")
print(f"Ventas: {len(df_ventas)} registros")

## 1. Suite Parametrizada

Crear una suite que pueda aplicarse a diferentes columnas.

In [ ]:
def crear_suite_numerica(context, nombre_suite, columna, min_val, max_val):
    """
    Crea una suite parametrizada para validar columnas numéricas.
    """
    suite = context.suites.add(gx.ExpectationSuite(name=nombre_suite))
    
    # No nulos
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(column=columna)
    )
    
    # Rango
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToBeBetween(
            column=columna,
            min_value=min_val,
            max_value=max_val
        )
    )
    
    suite.save()
    return suite

# Usar la función
context = gx.get_context(mode="ephemeral")
datasource = context.data_sources.add_pandas(name="ventas_ds")
asset = datasource.add_dataframe_asset(name="ventas")
batch_def = asset.add_batch_definition_whole_dataframe("batch_completo")

# Crear suites para diferentes columnas
suite_price = crear_suite_numerica(context, "validacion_price", "price", 0.01, 10000)
suite_quantity = crear_suite_numerica(context, "validacion_quantity", "quantity", 1, 100)

print(" Suites parametrizadas creadas")

## 2. Aplicar Suite a Múltiples Datasets

In [ ]:
# Validar price
val_def_price = context.validation_definitions.add(
    gx.ValidationDefinition(data=batch_def, suite=suite_price, name="val_price")
)
resultado_price = val_def_price.run(batch_parameters={"dataframe": df_ventas})

# Validar quantity
val_def_quantity = context.validation_definitions.add(
    gx.ValidationDefinition(data=batch_def, suite=suite_quantity, name="val_quantity")
)
resultado_quantity = val_def_quantity.run(batch_parameters={"dataframe": df_ventas})

print(f"Price: {'' if resultado_price.success else ''}")
print(f"Quantity: {'' if resultado_quantity.success else ''}")

## 3. Configuración con Diccionarios

In [ ]:
# Configuración centralizada
config_validaciones = {
    "price": {"min": 0.01, "max": 10000, "tipo": "float"},
    "quantity": {"min": 1, "max": 100, "tipo": "int"},
}

# Crear suites desde config
for columna, config in config_validaciones.items():
    suite_name = f"validacion_{columna}_config"
    suite = context.suites.add(gx.ExpectationSuite(name=suite_name))
    
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToBeBetween(
            column=columna,
            min_value=config["min"],
            max_value=config["max"]
        )
    )
    suite.save()

print(" Suites creadas desde configuración")

##  Ejercicio

Crea una función que genere una suite completa de calidad para cualquier dataset con columnas: id, fecha, valor_numerico.

In [ ]:
def crear_suite_generica(context, nombre, col_id, col_fecha, col_valor, min_valor, max_valor):
    # TU CÓDIGO AQUÍ
    pass

In [ ]:
context.build_data_docs()
context.open_data_docs()

##  Resumen

1.  Parametrización permite reutilizar lógica
2.  Funciones crean suites dinámicamente
3.  Configuraciones centralizadas facilitan mantenimiento
